# MachadoGPT



In [51]:
import os 
import pickle
import argparse

import numpy as np
import torch 
import torch.nn as nn
from torch.nn import functional as F

## Preparando os dados

A pasta `inputs/` contém o conjunto de obras do Machado de Assis dividas por estilo literário:
- poesia
- teato
- texto - Romance e Contos

O conteúdo desses arquivos consiste nas obras presentes em `dataset/` concatenadas para um único arquivo `.txt`



In [52]:
INPUT_DIR = 'inputs'
OUTPUT_DIR = 'inputs-bin'

### DataPreparer

Objetivos dessa etapa:
- Coletar todos os caracteres únicos que ocorrem no input - vocabulário
- Fazer um mapeamento entre caracteres e inteiros - O modelo utiliza um tokenizer a nível de caracter
- Criar os dados de treino (90%) e validação (10%)
- Salvar os dados (`train_data`, `val_data`), assim como os metados (em `meta.pkl`), no respectivo diretório em `inputs-bin/`

In [53]:
class DataPreparer:

    def prepare(self, file: str):

        file_path = os.path.join(INPUT_DIR, f"{file}.txt")

        output_dir = os.path.join(os.path.dirname(__file__), OUTPUT_DIR)
        if not os.path.exists(output_dir):
            os.makedirs(output_dir)

        output_path = os.path.join(output_dir, file)
        if not os.path.exists(output_path):
            os.makedirs(output_path)

        # get all the unique characters that occur in this text
        data, chars, vocab_size = self._get_vocab(file_path)
        print("all the unique characters:", ''.join(chars))
        print(f"vocab size: {vocab_size:,}")    

        # create a mapping from characters to integers
        stoi, itos = self._create_mapping(chars)

        # create the train and test splits
        train_data, val_data = self._split_data(data)

        # encode both to integers
        train_ids = self._encode(train_data, stoi)
        val_ids = self._encode(val_data, stoi)
        print(f"train has {len(train_ids):,} tokens")
        print(f"val has {len(val_ids):,} tokens")

        # export to bin files
        self._export_splits(train_ids, val_ids, output_path)

        # save the meta information as well, to help us encode/decode later
        self._save_metadata(chars, vocab_size, itos, stoi, output_path)


    def _get_vocab(self, file: str):
        with open(file, 'r') as f:
            data = f.read()

        chars = sorted(list(set(data)))
        vocab_size = len(chars)

        return data, chars, vocab_size


    def _create_mapping(self, chars: list[str]): 
        stoi = { ch:i for i, ch in enumerate(chars) }
        itos = { i:ch for i, ch in enumerate(chars) }
        return stoi, itos


    def _encode(self, s: str, stoi: dict):
        return [stoi[c] for c in s]


    def _decode(self, l: list[int], itos: dict):
        return [itos[i] for i in l]


    def _split_data(self, data: str):
        n = len(data)
        train_data = data[:int(n*0.9)]
        val_data = data[int(n*0.9):]

        return train_data, val_data


    def _export_splits(self, train_ids, val_ids, output_path):
        train_ids = np.array(train_ids, dtype=np.uint16)
        val_ids = np.array(val_ids, dtype=np.uint16)
        train_ids.tofile(os.path.join(output_path, 'train.bin'))
        val_ids.tofile(os.path.join(output_path, 'val.bin'))


    def _save_metadata(self, chars: list, vocab_size: int, itos: dict, stoi: dict, output_path):
        meta = {
            'chars': chars,
            'vocab_size': vocab_size,
            'itos': itos,
            'stoi': stoi,
        }
        with open(os.path.join(output_path, 'meta.pkl'), 'wb') as f:
            pickle.dump(meta, f)

## Modelo

In [54]:
INPUT_DIR = os.path.join(os.getcwd(), "inputs-bin")
OUT_DIR = 'out'

### Hiperparâmetros
1. `batch_size`: quantas sequências independentes vão ser processadas em paralelo.
2. `block_size`: Qual é o tamanho máximo de contexto das predições
3. `max_iters`: número total de iterações 
4. `eval_interval`: de quantas em quantas iterações o modelo é avaliado
5. `learning_rate`: determina o tamanho das atualizações que o modelo faz durante a otimização 
6. `eval_iters` = quantos batches são usados a cada round de avaliação
7. **`n_embd`**: tamanho do vetor que representa cada token internamente no modelo 
    - Quando um token entra no modelo, ele é convertido num vetor de {n_embd} números
8. **`n_head`** = Divide o mecanismo de self-attention em {cabeças} paralelas
    - Cada cabeça pode se especializar em capturar um tipo diferente de relação; Uma pode aprender dependências sintáticas de curto alcance (ex: sujeito-verbo), outra pode capturar relações de longo alcance, outra pode focar em pontuação, etc. É uma forma de dar diversidade representacional sem aumentar o custo total de computação (o custo total de attention é aproximadamente o mesmo, só é dividido entre as cabeças).
9. **`n_layer`**: Quantas vezes o bloco (self-attention + feedforward, com residual connections e layer norm) se repete em sequência.
10. `dropout`:  probabilidade de "desligar" aleatoriamente neurônios durante o treino como técnica de regularização para evitar overfitting

In [55]:
batch_size = 12       
block_size = 64       
max_iters = 10      
eval_interval = 500   
learning_rate = 3e-4 
eval_iters = 20
n_embd = 128
n_head = 8
n_layer = 8
dropout = 0.2

In [56]:
torch.manual_seed(1337) # seed para reprodução 
device = 'cuda' if torch.cuda.is_available() else 'cpu' # utiliza GPU se disponível 


### Carregando o Dataset

Essa etapa carrega os dados direto de `inputs-bin` de acordo com o tipo literário selecionado. 
São carregados os dados de treinamento e os metadados, dando acesso à:
- `vocab_size`
- `stoi`:       mapeia chars para inteiros 
- `itos`:       mapeia inteiros para chars
- `train_data`
- `val_data`



In [57]:
def load_dataset(tipo_literario: str):
    input_path = os.path.join(INPUT_DIR, tipo_literario)

    # carrega metadados
    with open(os.path.join(input_path, "meta.pkl"), "rb") as f:
        meta = pickle.load(f)

    vocab_size = meta["vocab_size"]
    stoi = meta["stoi"]
    itos = meta["itos"]

    # carrega train e val respeitando o dtype usado na escrita (uint16)
    train_arr = np.fromfile(os.path.join(input_path, "train.bin"), dtype=np.uint16)
    val_arr = np.fromfile(os.path.join(input_path, "val.bin"), dtype=np.uint16)

    train_data = torch.tensor(train_arr, dtype=torch.long)
    val_data = torch.tensor(val_arr, dtype=torch.long)

    return vocab_size, stoi, itos, train_data, val_data

In [58]:
tipo_literario = 'poesia'
input_path = os.path.join(INPUT_DIR, tipo_literario)
if not os.path.exists(os.path.join(input_path, "meta.pkl")):
    DataPreparer().prepare(tipo_literario)

vocab_size, stoi, itos, train_data, val_data = load_dataset(tipo_literario)

### Mapeamento de Inputs e Targets

A função seleciona `batch_size` subsequências de comprimento `block_size`,
extraídas em posições iniciais aleatórias do conjunto de dados especificado
(treino ou validação). Para cada subsequência de entrada, o alvo (target)
correspondente é a mesma sequência deslocada em uma posição à direita,
de modo que o modelo seja treinado a prever o próximo token a partir do
contexto observado (next-token prediction).

In [59]:
def get_batch(split: str):
    data = train_data if split == 'train' else val_data

    # {batch_size} inteiros aleatorios entre [0 : len(data) - block_size]
    ix = torch.randint(len(data) - block_size, (batch_size,))

    inputs = torch.stack([data[i:i+block_size] for i in ix])
    targets = torch.stack([data[i+1:i+1+block_size] for i in ix])
    # exemplo:
    # inputs:  [...0,4,5,9...]
    # targets: [...4,5,9,3...]

    inputs, targets = inputs.to(device), targets.to(device)

    return inputs, targets

### Estimativa da loss média em treino e validação.

Calcula uma aproximação da loss esperada para os conjuntos de treino e
validação, através da média de `eval_iters` mini-batches amostrados
aleatoriamente via `get_batch`. O uso da média sobre múltiplos batches
reduz a variância da estimativa em comparação a uma avaliação em um
único batch, fornecendo um sinal mais estável do desempenho do modelo.

A função opera em modo de avaliação (`model.eval()`) e sob o
decorador `@torch.no_grad()`, que desabilita o cálculo de gradientes,
reduzindo uso de memória e custo computacional já que nenhuma
atualização de pesos ocorre nesta etapa. Ao final, o modelo é
retornado ao modo de treino (`model.train()`).

1. **Modo treino** (`model.train()`, padrão): a cada forward pass, {`dropout`}% dos neurônios são zerados aleatoriamente. Isso é uma técnica de regularização - força a rede a não depender demais de neurônios específicos, melhorando a generalização.

2. **Modo avaliação** (`model.eval()`): o dropout é desativado. Todos os neurônios ficam ativos, e a saída passa a ser determinística (sem a aleatoriedade do "desligar" neurônios).

In [60]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

### Single Head of Causal Self-Attention

**obs**: `Causal` significa que cada token pode apenas olhar para trás


- **Query** (Q) — "o que este token está procurando". Representa a pergunta que cada token faz ao restante da sequência: "quais informações do contexto são relevantes para mim?"
- **Key** (K) — "o que este token oferece como identificador". Cada token expõe uma chave que descreve o tipo de informação que ele carrega, para ser comparada com as queries dos outros.
- **Value** (V) — "o conteúdo real que este token carrega". É a informação de fato transportada, que será agregada caso o token seja considerado relevante (alta afinidade Q·K).


#### **`nn.Linear(n_embd, head_size, bias=False)`**

Entrada: n_embd = 128 — dimensão do embedding de cada token.
Saída: head_size = 16 — dimensão do espaço projetado (K, Q ou V).
Pesos (W): matriz treinável de formato (16, 128), aplicada como y = x @ W.T. Sem bias, é uma combinação linear pura dos 128 valores de entrada, gerando 16 valores de saída. W começa aleatória e é ajustada via backpropagation, aprendendo a extrair de x a informação relevante para servir como key, query ou value.
Essa projeção acontece independentemente para cada token da sequência — é o que gera as matrizes K, Q, V usadas a seguir no cálculo de atenção.

#### **Trill**
Exemplo:

| Token | tok0 | tok1 | tok2 | tok3 | tok4  |
| -- | -- | -- | -- | -- | -- |
|tok0 | 0.9  | 0.3  | 0.5  | 0.1  | 0.4 |
|tok1 | 0.2  | 0.8 |  0.4 |  0.6  | 0.3 |
|tok2 | 0.1 | 0.3 | 0.7 | 0.5 | 0.2 |
|tok3 | 0.4 | 0.2 | 0.6 | 0.9 | 0.1 |
|tok4 | 0.3 | 0.5 | 0.2 | 0.4 | 0.8  |

tril (triangular inferior 5×5):

[ 1  0  0  0  0 ]  
[ 1  1  0  0  0 ]  
[ 1  1  1  0  0 ]  
[ 1  1  1  1  0 ]  
[ 1  1  1  1  1 ]  

Depois de `masked_fill(tril == 0, -inf)`, tudo acima da diagonal (posições futuras) vira -inf:

[  0.9   -inf  -inf  -inf  -inf ]  
[  0.2    0.8  -inf  -inf  -inf ]  
[  0.1    0.3   0.7  -inf  -inf ]  
[  0.4    0.2   0.6   0.9  -inf ]  
[  0.3    0.5   0.2   0.4   0.8 ]  


#### **Dropout**

Depois do softmax, wei é uma matriz de pesos de atenção (probabilidades, cada linha somando 1). O nn.Dropout(0.2) zera aleatoriamente 20% dessas conexões a cada forward pass durante o treino, reescalando as restantes para preservar a soma esperada.

Isso impede que o modelo "confie demais" numa única relação de atenção (ex: sempre olhar fortemente para o token anterior), forçando-o a distribuir a atenção de forma mais robusta entre várias posições — reduzindo overfitting. Em model.eval() (como na função estimate_loss), o dropout é desativado, e todos os pesos de atenção calculados permanecem ativos e determinísticos.


#### **Etapas do Forward:**
1. Projeção: `x` (B, T, n_embd) é projetado em Q e K (B, T, head_size),
via as camadas lineares aprendidas `query` e `key`.
2. Afinidades: os scores de atenção são obtidos por Q @ K^T, escalados
por 1/sqrt(head_size) (scaled dot-product attention), evitando que
produtos escalares grandes saturem o softmax.
3. Máscara causal: a matriz triangular `tril` impõe que a posição t só
possa atender a posições j <= t, preservando a causalidade necessária
para a previsão autoregressiva do próximo token.
4. Softmax: converte as afinidades mascaradas em pesos de atenção
(distribuição de probabilidade por linha).
5. Dropout: desativa aleatoriamente 20% dos pesos de atenção durante o
treino, como regularização.
6. Agregação: os values V (projeção de `x`) são combinados via média
ponderada pelos pesos de atenção (`wei @ v`), produzindo, para cada
token, uma representação contextualizada com base em si mesmo e nos
tokens anteriores.


In [61]:
class Head(nn.Module):

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # input of size (batch, time-step, channels)
        # output of size (batch, time-step, head size)
        B,T,C = x.shape
        k = self.key(x)   # (B,T,hs)
        q = self.query(x) # (B,T,hs)
        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2,-1) * k.shape[-1]**-0.5 # (B, T, hs) @ (B, hs, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T) 
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)
        # perform the weighted aggregation of the values
        v = self.value(x) # (B,T,hs)
        out = wei @ v # (B, T, T) @ (B, T, hs) -> (B, T, hs)
        return out

### **Multiple Heads of Self-Attention**

Em vez de calcular uma única atenção sobre todo o espaço de n_embd
dimensões, o mecanismo divide a computação em `num_heads` cabeças
independentes (instâncias de `Head`), cada uma operando em um subespaço
de dimensão `head_size` (onde head_size = n_embd / num_heads). Isso
permite que cada cabeça aprenda a capturar um tipo diferente de relação
entre os tokens (ex.: dependências sintáticas locais, relações de longo
alcance, etc.), sem aumentar o custo computacional total em relação a
uma única cabeça de dimensão n_embd.

- `self.proj`: camada linear que projeta a concatenação das saídas de
todas as cabeças (dimensão head_size * num_heads = n_embd) de volta
para o espaço de dimensão n_embd, permitindo combinar a informação
captada por cada cabeça.

#### **Forward**
1. Cada cabeça `h` processa `x` independentemente, produzindo uma saída
de formato (B, T, head_size).
2. `torch.cat(..., dim=-1)` concatena as saídas das `num_heads` cabeças
ao longo da última dimensão, recompondo um tensor de formato
(B, T, head_size * num_heads) = (B, T, n_embd).
3. `self.proj` mistura linearmente as informações vindas das diferentes
cabeças, permitindo que o modelo aprenda a combinar os diferentes
"pontos de vista" de atenção capturados por cada uma.
4. `self.dropout` é aplicado à saída da projeção, como regularização
adicional antes de seguir para a próxima camada da rede.

In [62]:
class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

### Feedforward Posicional
Rede feedforward posicional (position-wise), aplicada independentemente
a cada token, após o bloco de self-attention.

Enquanto a self-attention permite que os tokens troquem informação entre
si (olhando para o contexto), a FeedForward refina a representação de
cada token isoladamente, adicionando capacidade não-linear ao modelo.

Arquitetura (nn.Sequential):
1. `nn.Linear(n_embd, 4*n_embd)`: expande a dimensão por um fator de 4
(128 -> 512), aumentando temporariamente a capacidade representacional.
Esse fator 4x segue a convenção do paper original do Transformer.
2. `nn.ReLU()`: não-linearidade (ReLU(x) = max(0, x)). Sem ela, as duas
camadas lineares em sequência equivaleriam a uma única transformação
linear — é o que permite aprender padrões mais complexos.
3. `nn.Linear(4*n_embd, n_embd)`: projeta de volta à dimensão original
(512 -> 128), compatível com o restante do modelo.
4. `nn.Dropout(dropout)`: regularização, zerando 20% das saídas no treino.

In [63]:
class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),  # non-linear
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)




### Bloco transformer:
Uma etapa de "comunicação" (self-attention) seguida
de uma etapa de "computação" (feedforward), com conexões residuais e
normalização de camada (pre-LayerNorm).

Esse bloco é a unidade repetida `n_layer` vezes para formar o modelo
completo — empilhar vários blocos é o que dá profundidade à rede.


In [64]:
class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

### GPT Language Model

##### `__init__`

- `token_embedding_table`: para cada caractere/token do vocabulário (vocab_size entradas), guarda um vetor que representa seu "significado".
- `position_embedding_table`: para cada posição possível na sequência (0 a block_size-1), guarda um vetor que representa "isso está na posição t". Isso é necessário porque a self-attention, por si só, não tem noção de ordem, ela trata a sequência como um conjunto; a informação posicional precisa ser injetada explicitamente.
- `blocks`: Empilha n_layer blocos transformer em sequência 
- `ln_f`: normalização final, aplicada depois de todos os blocos, antes da camada de saída.
- `lm_head` ("language model head"): projeta o vetor de n_embd dimensões de volta para vocab_size dimensões - um número (logit) por caractere do vocabulário, indicando a "pontuação" de cada possível próximo token.

##### `_init_weights`

Em vez de deixar o PyTorch usar sua inicialização padrão, o modelo inicializa manualmente todos os pesos de camadas Linear e Embedding com uma distribuição normal de média 0 e desvio-padrão 0.02

##### `Forward` 

Etapas:
1. Embedding de token: cada índice inteiro de `idx` (B,T) é convertido
em seu vetor de embedding correspondente, via `token_embedding_table`,
resultando em (B,T,n_embd) — representa "o que é" cada token.
2. Embedding de posição: `position_embedding_table` gera um vetor para
cada posição de 0 a T-1, resultando em (T,n_embd) — representa
"onde" cada token está na sequência, informação que a self-attention
não capta sozinha.
3. Combinação: `tok_emb + pos_emb` soma os dois embeddings (com
broadcasting), produzindo a representação inicial (B,T,n_embd) de
cada token, já com contexto posicional embutido.
4. Blocos transformer: `x` atravessa os `n_layer` blocos empilhados,
cada um aplicando self-attention (comunicação entre tokens) seguida
de feedforward (computação por posição), com conexões residuais,
refinando progressivamente a representação de cada token conforme
o contexto.
5. Normalização final: `ln_f` estabiliza a escala dos valores da saída
do último bloco antes da projeção final.
6. Projeção para logits: `lm_head` projeta cada vetor de n_embd
dimensões para vocab_size dimensões, gerando um score (logit) por
caractere possível do vocabulário, em cada posição da sequência.
7. Cálculo da loss (condicional): se `targets` for fornecido, os
tensores são achatados para (B*T, C) e (B*T,) e a cross-entropy
mede a distância entre os logits previstos e os tokens esperados;
se `targets` for None (modo geração), a loss não é calculada.


##### `Generate`

Gera novos tokens de forma autoregressiva, a partir de um contexto inicial.

Etapas (repetidas `max_new_tokens` vezes):
1. Recorte do contexto: `idx_cond = idx[:, -block_size:]` mantém apenas
os últimos `block_size` tokens da sequência, já que o modelo (via
position_embedding_table e a máscara causal `tril`) só suporta
sequências de até `block_size` tokens.
2. Forward pass: `logits, loss = self(idx_cond)` executa o modelo sem
`targets`, retornando apenas os logits (loss = None).
3. Foco no último passo: `logits = logits[:, -1, :]` extrai somente os
logits referentes à última posição da sequência — a previsão de
"qual é o próximo token", dado todo o contexto processado.
4. Softmax: `probs = F.softmax(logits, dim=-1)` converte os logits em
uma distribuição de probabilidade sobre o vocabulário.
5. Amostragem: `idx_next = torch.multinomial(probs, num_samples=1)`
sorteia um token dessa distribuição — não escolhe necessariamente o
mais provável (estratégia gulosa/greedy), introduzindo variedade
na geração.
6. Concatenação: `idx = torch.cat((idx, idx_next), dim=1)` anexa o
token gerado ao final da sequência, que cresce a cada iteração e
passa a compor o contexto usado na próxima rodada.

In [65]:
class GPTLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)

        # better init, not covered in the original GPT video, but important, will cover in followup video
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

## Inicialização e Setup
Script de treinamento do modelo GPTLanguageModel, com suporte a checkpoint
(salvar/retomar) e geração de texto ao final.

Etapas:

1. Setup inicial: define diretórios de saída (`out_dir`) e checkpoint
   (`checkpoint_dir`), inicializa `best_val_loss` com um valor alto e
   `iter_num` em 0, e instancia o modelo `GPTLanguageModel()`.

2. Carregamento de checkpoint (condicional): se `init_from == 'resume'`,
   carrega o checkpoint salvo (`ckpt.pt`) via `torch.load`, restaurando os
   pesos do modelo (`model.load_state_dict`), o número da iteração onde o
   treino parou (`iter_num`) e a melhor loss de validação já observada
   (`best_val_loss`), permitindo continuar o treino de onde parou. Caso
   contrário, o modelo é treinado do zero.

3. Envio ao dispositivo e contagem de parâmetros: move o modelo para
   `device` (`m = model.to(device)`) e imprime o número total de parâmetros
   treináveis, em milhões.

4. Otimizador: instancia `AdamW` com `learning_rate`; se estiver retomando
   o treino, também restaura o estado interno do otimizador
   (`optimizer.load_state_dict`), preservando momentos acumulados.

5. Loop de treinamento (`max_iters` iterações a partir de `iter_num`):
   a. Avaliação periódica: a cada `eval_interval` iterações, calcula a
      loss média em treino e validação via `estimate_loss()`. Se a loss de
      validação melhorar, salva um checkpoint (pesos do modelo, estado do
      otimizador, iteração atual e melhor loss) em `checkpoint_dir` —
      garantindo que apenas o melhor modelo observado seja persistido.
   b. Passo de treino: amostra um batch (`get_batch('train')`), calcula
      logits e loss via `model(xb, yb)`, zera os gradientes anteriores
      (`optimizer.zero_grad`), calcula os novos gradientes (`loss.backward()`)
      e atualiza os pesos (`optimizer.step()`).

6. Geração final: a partir de um contexto inicial vazio (tensor de zeros),
   gera 2000 novos tokens autoregressivamente via `m.generate`, decodifica
   os índices de volta para caracteres, e salva o texto resultante em
   `{out_dir}/output.txt`.

In [66]:
# inicializacao = ['scratch', 'resume'])
# tipo literarios =['texto', 'poesia', 'teatro'])

init_from = 'resume'
tipo_literario = 'poesia'

In [67]:
out_dir = os.path.join(OUT_DIR, tipo_literario)
checkpoint_dir = os.path.join('checkpoints', tipo_literario)
best_val_loss = 1e9
checkpoint = None
iter_num = 0

model = GPTLanguageModel()

if init_from == 'resume':
    print(f"Resuming training from {checkpoint_dir}")
    ckpt_path = os.path.join(checkpoint_dir, 'ckpt.pt')
    checkpoint = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(checkpoint['model'])
    iter_num = checkpoint['iter_num']
    best_val_loss = checkpoint['best_val_loss']
    print(f'iter_num: {iter_num}')
    print(f'best_val_loss: {best_val_loss}')
else:
    print("Initializing a new model from scratch")

m = model.to(device)
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

if init_from == 'resume':
    optimizer.load_state_dict(checkpoint['optimizer']) # TypeIgnore
    checkpoint = None



Resuming training from checkpoints/poesia
iter_num: 12000
best_val_loss: 1.6872133016586304
1.622392 M parameters


In [70]:
m.eval()
context = torch.zeros((1, 1), dtype=torch.long, device=device)

decode = lambda l: [itos[i] for i in l]
out = ''.join(decode(m.generate(context, max_new_tokens=2000)[0].tolist()))
print(out)


A pálida muito, te que ceenjanho,
E do tecar, a frioso e e calve,
De abroncada co fran descobre a e estrecido
Como um agre, que pois vere mais sumprro,
Nor estre algia é trarrínvel
Cheia tre polvo de ti perdo,
Ia a frec, rastendo, a mens,
Vinga a a o vitório,
Amporo os sucenso e não é perdor.
Aro luz
Da o responder, de eu andar em desses,
E vaga do mal.
Anjo, reio menem quaier-se que; e sas as bantas
Em que em crava, a Rano,
Te precio
Goia as jovelos indo e more;
Que abe fica em fronte,
CSIIV A ia perfetunto a queles porve deram
Armador-te de um peito de que o
chora aoverso da paro o amor.
XLIV, Fescas
Em sonhais no precedo os ascrigados,
Musta e à porta que deis flutu,
E os camos úfareas das idesas
Sobre, cresce de famos e a silária os tremeluãos!
Mes alguns depela e algo dessos
Em criso não se mas; presia ambo
De cansadr um trivo que se venens,
Ele corta abasta;
Sobe um andor apare-sa do me escrit,
Carr, de espravo o ensange de avesta
Gualestes,
E à suprensas globera loto
De dessor 

In [ ]:
for iter in range(iter_num, iter_num + max_iters + 1):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

        if losses['val'] < best_val_loss:
            best_val_loss = losses['val']
            os.makedirs(checkpoint_dir, exist_ok=True)
            torch.save({
                'model': model.state_dict(),
                'optimizer': optimizer.state_dict(),
                'iter_num': iter,
                'best_val_loss': best_val_loss,
            }, os.path.join(checkpoint_dir, 'ckpt.pt'))
            print(f"checkpoint salvo em {checkpoint_dir}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)

out = ''.join(decode(m.generate(context, max_new_tokens=2000)[0].tolist()))
print(out)
# os.makedirs(out_dir, exist_ok=True)
# with open(f'{out_dir}/ouput.txt', 'w') as f:
#     f.write(out)



step 12000: train loss 1.4111, val loss 1.6847
checkpoint salvo em checkpoints/poesia


KeyboardInterrupt: 